# TRIADE — LE KIT D'ENTRAÎNEMENT COLAB — **0 CLIC** (v1062)

La campagne d'**entraînement** de Qwen2.5-Coder-7B avec TRIADE : **QLoRA SFT** sur les 17 VERTS du corpus, puis **LA MÊME BATTERIE** que la campagne 105 au modèle affiné — le `--lora` du delta sur le GGUF Q6_K byte-byte. `Exécution → Tout exécuter` — c'est tout. Au terme le zip (l'adapter + la mesure) + le rapport vivent au `/test` du coffre. **Le T4 est la voie conseillée** (le plan gelé du kit est LA LOI du run ; l'amendement terrain.11 du kit — la perte à `logits_to_keep`, les logits du prompt jamais alloués — soigne le refus mémoire mesuré au run du 28/11, le plan et le corpus demeurent byte-byte).

## 1 · LE POSTE COLAB — le verdict de l'environnement + LE PLAN DU RUN

In [ ]:
# CELLULE 1 — LE POSTE COLAB : le verdict de l'environnement + LE PLAN DU RUN
# (v106 — LA CAMPAGNE D'ENTRAÎNEMENT : QLoRA SFT puis LA MÊME BATTERIE que la 105).
# v105 — la reprise des tests ratés du v104 (l'Entrée 451) : le poste décide
# SEUL le bord de la batterie (GPU ou
# CPU), la mesure imprimée TELLE QUELLE (AUCUN chiffre fabriqué). LE RAPPORT
# DU POSTE naît ici (la pièce nouvelle du v104 : le verdict d'environnement +
# l'horloge de la batterie y vivront) — la cellule 7 le déposera au /test du
# coffre avec le zip de preuves. Les gardes honnêtes refusent AVANT la
# marche, jamais deux heures après (fail-closed, l'écart nommé).
import datetime
import pathlib
import subprocess

import shutil
import sys

gpu_nom, gpu_vram = None, None
try:
    import torch
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        gpu_nom, gpu_vram = p.name, p.total_memory / 2**30
except Exception as e:
    print(f"[attention] torch introuvable ou GPU muet : {e.__class__.__name__}")

ram = None
for ligne in open("/proc/meminfo", encoding="utf-8"):
    if ligne.startswith("MemTotal"):
        ram = int(ligne.split()[1]) / 2**20
        break
disque = shutil.disk_usage("/").free / 2**30

DEBUT_RUN = datetime.datetime.now(datetime.timezone.utc)
RAPPORT = pathlib.Path("/content/rapport-poste.txt")


def rapport(t):
    with open(RAPPORT, "a", encoding="utf-8") as f:
        f.write(t + "\n")


RAPPORT.parent.mkdir(parents=True, exist_ok=True)
RAPPORT.write_text("", encoding="utf-8")
rapport(f"TRIADE — LE RAPPORT DU POSTE — kit Colab v106 ENTRAÎNEMENT (0 clic) — {DEBUT_RUN.isoformat()}")

torch_version = None
try:
    import torch as _torch
    torch_version = _torch.__version__
except Exception:
    pass

MODE_GPU = bool(gpu_vram)
QUANT = "Q6_K" if MODE_GPU else "Q4_K_M"
if MODE_GPU:
    r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                        "--format=csv,noheader"], capture_output=True, text=True)
    if r.returncode == 0:
        print(f"[poste] nvidia-smi : {r.stdout.strip()}")

ligne_gpu = f"{gpu_nom} ({gpu_vram:.1f} Gio VRAM)" if gpu_vram else "AUCUN — le bord CPU"
print(f"[poste] le GPU    : {ligne_gpu}")
if ram:
    print(f"[poste] la RAM   : {ram:.1f} Gio")
print(f"[poste] le disque : {disque:.1f} Gio libres")
print(f"[poste] le Python : {sys.version.split()[0]} · torch {torch_version}")
print()
print(f"[plan] LE BORD DU RUN : {'GPU' if MODE_GPU else 'CPU'} — le GGUF {QUANT} "
      f"({'6,25 Go' if MODE_GPU else '4,68 Go — le plus petit du kit'}).")
if MODE_GPU:
    print("[plan] la release cudart prébattie + -ngl 99 — la batterie complète court seule (T4 et plus).")
    print("[plan] l'estimation du kit : de quelques dizaines de minutes à ~2 heures au MODE-B sur T4 selon la charge.")
else:
    print("[plan] la release ubuntu-x64 NUE, AUCUN -ngl — le bord natif du CPU.")
    print("[plan] L'ESTIMATION HONNÊTE du bord CPU : la campagne y prend PLUSIEURS HEURES")
    print("[plan] (la mesure seule dira — la voie GPU est la voie conseillée ; le run court quand même).")
print("[plan] LE RUN D'ENTRAÎNEMENT : les 17 VERTS du corpus au QLoRA SFT (le plan gelé")
print("[plan] v106), PUIS LA MÊME BATTERIE que la 105 au modèle affiné (le --lora du")
print("[plan] delta sur le GGUF Q6_K byte-byte — la seule différence du bord, nommée).")
print("[plan] L'ADAPTER est téléversé au coffre AVANT l'éval (l'œuvre sauvée d'abord).")
for t in (f"[poste] le GPU : {ligne_gpu}",
          f"[poste] la RAM : {ram:.1f} Gio" if ram else "[poste] la RAM : ?",
          f"[poste] le disque : {disque:.1f} Gio libres",
          f"[poste] le Python : {sys.version.split()[0]} · torch {torch_version}",
          f"[plan] le bord du run : {'GPU' if MODE_GPU else 'CPU'} (le T4 est LA VOIE conseillée)",
          "[plan] le run d'entraînement v106 — QLoRA SFT puis la batterie de la 105 (le plan gelé"):
    rapport(t)

if disque < 40:
    raise SystemExit(f"DISQUE-INSUFFISANT : {disque:.1f} Gio libres — le modèle 4 bits + le GGUF "
                     "+ l'adapter + le cache HF en prennent ~25 ; change de runtime (le refus vient AVANT la marche).")
if not MODE_GPU:
    mem_disp = None
    for ligne in open("/proc/meminfo", encoding="utf-8"):
        if ligne.startswith("MemAvailable"):
            mem_disp = int(ligne.split()[1]) / 2**20
            break
    if mem_disp is not None:
        print(f"[poste] la RAM disponible : {mem_disp:.1f} Gio")
        rapport(f"[poste] la RAM disponible : {mem_disp:.1f} Gio")
        if mem_disp < 5.5:
            raise SystemExit(f"RAM-INSUFFISANTE au bord CPU : {mem_disp:.1f} Gio disponibles — le GGUF "
                             f"{QUANT} (4,68 Go) ne tient pas ; le runtime GPU est la voie.")


## 2 · LES DÉPENDANCES — le harnais + l'entraînement (peft, bitsandbytes)

In [ ]:
# CELLULE 2 — LES DÉPENDANCES DU HARNAIS : le pip du clic unique reconduit
# (COMMENCER-ICI.bat du kit poste : llguidance + tokenizers + keystone-engine + capstone ;
#  le MODE-A en sus exige torch + transformers — Colab les porte déjà.)
import subprocess
import sys

rep = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet",
                      "llguidance", "tokenizers", "keystone-engine", "capstone",
                      "peft", "bitsandbytes", "accelerate", "gguf"],
                     capture_output=True, text=True)
if rep.stdout:
    print(rep.stdout[-800:], end="")
if rep.stderr:
    print(rep.stderr[-800:], end="")
if rep.returncode != 0:
    raise SystemExit(f"PIP-ROUGE (code {rep.returncode}) — vérifie la connexion, puis relance la cellule.")

import capstone
import keystone
import llguidance
import tokenizers

import peft, bitsandbytes, accelerate, gguf
print(f"[deps] peft {peft.__version__} · bitsandbytes {bitsandbytes.__version__} · "
      f"accelerate {accelerate.__version__} · gguf OK — l'entraînement armé · "
      f"llguidance {getattr(llguidance, '__version__', '?')} · "
      f"tokenizers {tokenizers.__version__} · "
      f"keystone {getattr(keystone, '__version__', 'OK')} · "
      f"capstone {capstone.__version__} — VERT")


## 3 · LES DEUX KITS AU COFFRE — **0 CLIC** : le poste v101 (l'instrument) + le kit d'entraînement v106 (le corpus + le plan gelé)

In [ ]:
# CELLULE 3 — LES DEUX KITS AU COFFRE, 0 CLIC : le relais embarqué tire LE KIT
# POSTE v101 TEL QUEL (l'instrument épreuvé de la batterie — byte-byte la
# campagne 105) PUIS LE KIT D'ENTRAÎNEMENT v106 (le corpus + le plan gelé) —
# chacun à SON ID Drive et SON empreinte ÉPINGLÉS (KIT-DIVERGENT le refus
# nommé, fail-closed). AUCUN montage du Drive, AUCUNE fenêtre, AUCUN clic.
# LE JETON RELAIS du labo (le /Tokens du coffre, la voie éprouvée D263).
# LA COPIE AVEUGLE : le dépôt git est PUBLIC — sa copie porte les TROIS
# chaînes VIDES (IDENTIFIANTS-ABSENTS le refus nommé) ; la copie vivante vit
# au /livrables du coffre.
import hashlib
import json
import pathlib
import shutil
import time
import urllib.parse
import urllib.request
import zipfile

# LES IDENTIFIANTS DU COFFRE (la voie 0 clic — aucune fenêtre, aucun clic) :
CLIENT_ID = ""
CLIENT_SECRET = ""
REFRESH_TOKEN = ""

# LES DEUX KITS ÉPINGLÉS (le plan gelé PLAN-ENTRAINEMENT-COLAB-V106 § 5) :
KITS = [
    {   # l'instrument de la batterie — le v102 (moteur bivalent, terrain.10)
        "nom": "TRIADE_kit_campagne_poste_v102.zip",
        "sha256": "60962d74a06476b5746112d66ac94edfdf5f8b19d475cda30f5efe7e05d431b7",
        "id_drive": "1eEBGMND8d9pB2-41ybyXmoNpTdg-pkRz",
        "dest": pathlib.Path("/content/kit"),
    },
    {   # le corpus + le plan gelé + L'AMENDEMENT TERRAIN.11 de l'entraînement
        "nom": "TRIADE_kit_entrainement_v1061.zip",
        "sha256": "f5a2382b0204d9872a1f44695bfa1f83c9b79adc53754102924c61e144a64b1f",
        "id_drive": "1oFnEsGy6JV_JRw2h6MugHurxfeNCuH3T",
        "dest": pathlib.Path("/content/kit-entrainement"),
    },
]

if not (CLIENT_ID and CLIENT_SECRET and REFRESH_TOKEN):
    raise SystemExit("IDENTIFIANTS-ABSENTS : la présente est LA COPIE AVEUGLE du dépôt "
                     "public — les identifiants n'y vivent jamais. La copie vivante vit "
                     "au /livrables du coffre (Entrainement-Colab-v106.ipynb).")


def sha256f(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for morceau in iter(lambda: f.read(1 << 20), b""):
            h.update(morceau)
    return h.hexdigest()


def jeton_relais(essais=3):
    """Le jeton frappé au relais — la voie éprouvée D263, les ratés réessayés."""
    data = urllib.parse.urlencode({
        "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET,
        "refresh_token": REFRESH_TOKEN, "grant_type": "refresh_token"}).encode()
    derniere = None
    for i in range(essais):
        try:
            req = urllib.request.Request("https://oauth2.googleapis.com/token", data=data)
            with urllib.request.urlopen(req, timeout=30) as r:
                return json.loads(r.read().decode("utf-8"))["access_token"]
        except Exception as e:
            derniere = e
            print(f"[coffre] le relais a manqué ({e.__class__.__name__}) — "
                  f"le réessai {i + 2}/{essais} dans 10 s")
            time.sleep(10)
    raise SystemExit(f"JETON-ROUGE : le relais n'a pas donné son jeton "
                     f"({derniere.__class__.__name__}) — le journal ci-dessus TEL QUEL.")


def tirer_la_piece(tok, id_drive, nom):
    """La pièce tirée du coffre à l'ID épinglé — le compte à voix haute."""
    req = urllib.request.Request(
        f"https://www.googleapis.com/drive/v3/files/{id_drive}?alt=media",
        headers={"Authorization": f"Bearer {tok}"})
    dest = pathlib.Path("/content") / nom
    with urllib.request.urlopen(req, timeout=600) as r, open(dest, "wb") as f:
        total = int(r.headers.get("Content-Length") or 0)
        vu = 0
        while True:
            morceau = r.read(1 << 20)
            if not morceau:
                break
            f.write(morceau)
            vu += len(morceau)
            if total:
                print(f"[coffre] {nom} : {100 * vu // total} %", end="\r")
    print()
    return dest


print("[coffre] LE JETON RELAIS se frappe — AUCUNE fenêtre, AUCUN clic (la voie 0 clic).")
tok = jeton_relais()
for kit in KITS:
    zip_kit = tirer_la_piece(tok, kit["id_drive"], kit["nom"])
    empreinte = sha256f(zip_kit)
    print(f"[coffre] {kit['nom']} : {zip_kit.stat().st_size} o — sha256 {empreinte[:16]}…")
    if empreinte != kit["sha256"]:
        raise SystemExit(f"KIT-DIVERGENT : {kit['nom']} — {empreinte} != {kit['sha256']} "
                         "(l'empreinte épinglée) — le kit NE COURT PAS (fail-closed).")
    if kit["dest"].exists():
        shutil.rmtree(kit["dest"])
    with zipfile.ZipFile(zip_kit) as z:
        z.extractall(kit["dest"])
    pieces = [p for p in kit["dest"].rglob("*") if p.is_file()]
    print(f"[coffre] l'empreinte recoupée VERT — extrait à {kit['dest']} : {len(pieces)} pièces")


## 4 · LE GATE DE L'ENTRAÎNEMENT — le manifeste recoupé, LE COMPTE DES VERTS, l'enveloppe du moteur

In [ ]:
# CELLULE 4 — LE GATE DE L'ENTRAÎNEMENT : le manifeste du kit recoupé, LE
# COMPTE DES VERTS (17 — la loi des recettes), l'enveloppe du moteur importée
# (le MÊME langage que la batterie), le tokenizer du modèle chargé. Chaque
# écart = refus nommé AVANT la marche (fail-closed).
import hashlib
import json
import pathlib
import sys

DOS_KIT_ENT = pathlib.Path("/content/kit-entrainement")
DOS_KIT = pathlib.Path("/content/kit")

# 1 · le manifeste du kit — chaque pièce recoupée à son empreinte
manifeste = json.loads((DOS_KIT_ENT / "MANIFESTE-KIT-ENTRAINEMENT.json").read_text("utf-8"))
for piece in manifeste["pieces"]:
    p = DOS_KIT_ENT / piece["chemin"]
    if not p.exists():
        raise SystemExit(f"PIÈCE-ABSENTE : {piece['chemin']} — le kit est incomplet.")
    h = hashlib.sha256(p.read_bytes()).hexdigest()
    if h != piece["sha256"]:
        raise SystemExit(f"PIÈCE-DIVERGENTE : {piece['chemin']} — {h} != {piece['sha256']}")
print(f"[gate] le manifeste recoupé : {len(manifeste['pieces'])} pièces VERTES.")

# 2 · le plan gelé présent (la loi du kit)
if not (DOS_KIT_ENT / "PLAN-ENTRAINEMENT-COLAB-V106.md").exists():
    raise SystemExit("PLAN-ABSENT : le plan gelé vit au kit — AUCUNE mesure sans lui.")
print(f"[gate] le plan gelé : sha256 {manifeste['sha_plan']} (consigné au manifeste).")

# 3 · LE COMPTE DES VERTS (la loi des recettes : les VERTS seuls)
verts, total = [], 0
with open(DOS_KIT_ENT / "dataset" / "train.jsonl", encoding="utf-8") as f:
    for ligne in f:
        d = json.loads(ligne)
        total += 1
        if d["verdict"] == "VERT":
            verts.append(d)
ATTENDUS = manifeste["verts_attendus"]
print(f"[gate] le corpus : {total} triples — {len(verts)} VERTS (les REFUS exclus).")
if len(verts) != ATTENDUS:
    raise SystemExit(f"COMPTE-VERTS-DIVERGENT : {len(verts)} verts != {ATTENDUS} attendus "
                     "(le manifeste) — le corpus a changé, le plan ne court plus.")

# 4 · l'enveloppe du moteur — la pièce vivante importée (jamais une copie)
sys.path.insert(0, str(DOS_KIT))
from moteur_campagne import enveloppe  # noqa: E402
e = enveloppe("ESSAI")
assert e.startswith("<|im_start|>user\n") and e.endswith("<|im_start|>assistant\n")
print("[gate] l'enveloppe du moteur importée — le MÊME langage que la batterie (l'Entrée 451).")

# 5 · le tokenizer du modèle
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct")
probe = tok(enveloppe("probe"), add_special_tokens=False)["input_ids"]
print(f"[gate] le tokenizer vivant : {len(tok)} jetons au vocabulaire, la sonde {len(probe)} ids.")
print("[gate] LE GATE DE L'ENTRAÎNEMENT EST VERT — la cellule 5 (LE SFT) peut courir.")


## 5 · L'ENTRAÎNEMENT — QLoRA SFT sur les 17 VERTS (la perte sur la réponse seule, l'enveloppe du moteur, les hyperparams du plan gelé)

In [ ]:
# CELLULE 5 — L'ENTRAÎNEMENT QLoRA SFT : les 17 VERTS du corpus, la perte sur
# LA RÉPONSE SEULE, l'enveloppe VERBATIM du moteur, les hyperparams du plan
# gelé (§ 3) — chaque pas imprimé (le journal vit en direct, la courbe au
# registre). Les longs exemples d'ABORD (l'échec OOM vient tôt, jamais à la
# fin) ; EXEMPLE-TROP-LONG et EXEMPLE-LONG-OOM les refus nommés.
# L'AMENDEMENT TERRAIN.11 (AMENDEMENT-V106-TERRAIN11.md du kit) : la perte
# demandée à logits_to_keep (les logits du prompt JAMAIS alloués — le T4 ne
# les porte pas, mesuré au refus du 28/11 : 7,15 Gio pour la copie fp32 des
# logits pleins) puis conduite par tronçons de 1024 rangées — LA MÊME
# MATHÉMATIQUE (la preuve au labo : la même perte au ulp du fp32, le même
# signal), le plan gelé demeure LA LOI.
import json
import os
import pathlib
import time

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import torch
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model
from torch.utils.checkpoint import checkpoint
from transformers import (AutoModelForCausalLM, BitsAndBytesConfig,
                          get_cosine_schedule_with_warmup)

import sys
sys.path.insert(0, "/content/kit")
from moteur_campagne import enveloppe  # noqa: E402  (la pièce vivante)

# ── les constantes du plan gelé (§ 3) ──
MODELE = "Qwen/Qwen2.5-Coder-7B-Instruct"
SEED = 20261128
EPOQUES, ACCUM, LR = 3, 4, 1.5e-5
MAX_LEN = 13000          # AUCUNE troncature (le plan gelé § 2)
R, ALPHA, DROPOUT = 16, 32, 0.05
CIBLES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

torch.manual_seed(SEED)
DOS_ADAPTER = pathlib.Path("/content/adapter-entrainement-v106")
REGISTRE = pathlib.Path("/content/registre-entrainement-v106.json")

# ── les données : les VERTS seuls, l'enveloppe du moteur ──
verts = []
with open("/content/kit-entrainement/dataset/train.jsonl", encoding="utf-8") as f:
    for ligne in f:
        d = json.loads(ligne)
        if d["verdict"] == "VERT":
            verts.append(d)
print(f"[sft] {len(verts)} VERTS — la tokenisation court (l'enveloppe du moteur)…")
exemples = []
for d in verts:
    p = tok(enveloppe(d["contrat"]), add_special_tokens=False)["input_ids"]
    c = tok(d["reponse"] + "<|im_end|>", add_special_tokens=False)["input_ids"]
    if len(p) + len(c) > MAX_LEN:
        raise SystemExit(f"EXEMPLE-TROP-LONG : {d['triple']} à {len(p) + len(c)} jetons > "
                         f"{MAX_LEN} — le plan gelé refuse la troncature (fail-closed).")
    exemples.append({"triple": d["triple"], "p": p, "c": c})
exemples.sort(key=lambda e: -(len(e["p"]) + len(e["c"])))   # les longs d'abord
l_max = max(len(e["p"]) + len(e["c"]) for e in exemples)
print(f"[sft] le plus long exemple : {l_max} jetons ; le plus court : "
      f"{min(len(e['p']) + len(e['c']) for e in exemples)} — AUCUNE troncature.")

# ── le modèle 4 bits (le T4 porte le fp16, pas le bf16) ──
print("[sft] le modèle se charge en 4 bits NF4 (double quantification)…")
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_compute_dtype=torch.float16,
                         bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(
    MODELE, quantization_config=bnb, device_map={"": 0},
    torch_dtype=torch.float16, attn_implementation="sdpa")
model.gradient_checkpointing_enable()
model.enable_input_require_grads()
model.config.use_cache = False
lora_cfg = LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=DROPOUT, bias="none",
                      task_type="CAUSAL_LM", target_modules=CIBLES)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()

# ── la boucle : batch 1 × accumulation 4, la perte sur la réponse seule ──
params = [p for p in model.parameters() if p.requires_grad]
opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0)
n_pas = len(exemples) * EPOQUES
opt_pas = n_pas // ACCUM
sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * opt_pas)), opt_pas)
courbe, t0 = [], time.time()
print(f"[sft] LA MARCHE : {EPOQUES} époques × {len(exemples)} exemples = {n_pas} pas "
      f"({opt_pas} pas d'optimiseur) — le journal vit en direct ci-dessous.")

# ── l'amendement terrain.11 : la CE par tronçons (la mémoire du T4) ──
TRONCON = 1024


def ce_troncon(logits_bf16, cibles):
    return F.cross_entropy(logits_bf16.float(), cibles, reduction="sum")


model.train()
try:
    for epo in range(EPOQUES):
        for i, e in enumerate(exemples):
            ids = e["p"] + e["c"]
            t = torch.tensor([ids], device="cuda")
            cibles = torch.tensor(e["c"], device="cuda")
            out = model(input_ids=t, logits_to_keep=len(e["c"]) + 1)
            lg = out.logits[0, :-1, :]   # [C, V] — prédit exactement les cibles
            somme = torch.zeros((), device="cuda")
            for j in range(0, lg.shape[0], TRONCON):
                somme = somme + checkpoint(ce_troncon, lg[j:j + TRONCON],
                                           cibles[j:j + TRONCON], use_reentrant=False)
            perte = somme / len(e["c"])   # le « mean » aux non-ignorés (la même perte)
            (perte / ACCUM).backward()
            val = float(perte)
            courbe.append({"pas": len(courbe) + 1, "epoque": epo + 1,
                           "triple": e["triple"], "perte": round(val, 5)})
            if len(courbe) % ACCUM == 0:
                torch.nn.utils.clip_grad_norm_(params, 1.0)
                opt.step()
                sched.step()
                opt.zero_grad()
            if len(courbe) % 4 == 0 or len(courbe) == n_pas:
                print(f"[sft] époque {epo + 1}/{EPOQUES} · pas {len(courbe)}/{n_pas} · "
                      f"perte {val:.4f} · {time.time() - t0:.0f} s", flush=True)
except torch.cuda.OutOfMemoryError:
    raise SystemExit(f"EXEMPLE-LONG-OOM : le T4 n'a pas tenu l'exemple "
                     f"{e['triple']} ({len(ids)} jetons) — le plan gelé nomme le refus ; "
                     f"la voie de repli (l'exclusion nommée) est AU LABO, jamais au bord.")
model.eval()

# ── l'adapter sauvé + le registre né (la pièce d'œuvre du run) ──
model.save_pretrained(str(DOS_ADAPTER))
duree = time.time() - t0
REGISTRE.write_text(json.dumps({
    "modele": MODELE, "methode": "QLoRA-SFT", "quantisation": "NF4-double-fp16",
    "lora": {"r": R, "alpha": ALPHA, "dropout": DROPOUT, "cibles": CIBLES},
    "optimisation": {"lr": LR, "horloge": "cosine+5%", "epoques": EPOQUES,
                     "accumulation": ACCUM, "clip": 1.0, "graine": SEED,
                     "max_len": MAX_LEN, "troncature": "AUCUNE"},
    "amendement": {"terrain": "terrain.11", "perte": "logits_to_keep + CE par "
                   "tronçons de 1024 — la réponse seule, LA MÊME MATHÉMATIQUE "
                   "(la preuve au bac : la perte au ulp du fp32, EXACTEMENT "
                   "égale au vrai modèle, le même signal)"},
    "donnees": {"verts": len(exemples), "jetons_prompt": sum(len(e["p"]) for e in exemples),
                "jetons_reponse": sum(len(e["c"]) for e in exemples),
                "exemple_max": l_max},
    "pas": n_pas, "duree_s": round(duree, 1),
    "perte_premier": courbe[0]["perte"], "perte_dernier": courbe[-1]["perte"],
    "courbe": courbe,
}, ensure_ascii=False, indent=2), encoding="utf-8")
taille = sum(p.stat().st_size for p in DOS_ADAPTER.rglob("*") if p.is_file())
print(f"[sft] L'ENTRAÎNEMENT EST CLOSE : {n_pas} pas en {duree / 60:.1f} min — "
      f"perte {courbe[0]['perte']:.4f} → {courbe[-1]['perte']:.4f}.")
print(f"[sft] l'adapter : {DOS_ADAPTER} ({taille / 2**20:.1f} Mio) — "
      f"le registre : {REGISTRE} (la courbe y vit).")


## 6 · LE POST-ENTRAÎNEMENT — l'adapter converti au GGUF **puis téléversé au coffre AVANT l'éval** (l'œuvre sauvée d'abord)

In [ ]:
# CELLULE 6 — LE POST-ENTRAÎNEMENT : l'adapter converti au GGUF (le
# convertisseur du dépôt llama.cpp, à la MÊME release que le binaire du
# serving), puis LA PIÈCE TÉLÉVERSÉE AU COFFRE AVANT L'ÉVALUATION (le plan
# gelé § 4 : l'œuvre vit au coffre quand l'éval court — aucune perte si le
# runtime meurt en batterie). ADAPTER-CONVERSION-ROUGE le refus nommé.
import json
import pathlib
import subprocess
import sys
import time
import urllib.request

DOS_ADAPTER = pathlib.Path("/content/adapter-entrainement-v106")
GGUF_ADAPTER = pathlib.Path("/content/adapter-entrainement-v106.gguf")
REGISTRE = pathlib.Path("/content/registre-entrainement-v106.json")
CONVERTISSEUR = pathlib.Path("/content/convert_lora_to_gguf.py")

# 1 · le convertisseur — à la release du serving (la MÊME, la cohérence du bord)
def tag_de_la_release():
    with urllib.request.urlopen("https://api.github.com/repos/ggml-org/llama.cpp/releases/latest",
                                timeout=60) as r:
        return json.loads(r.read().decode())["tag_name"]

TAG = tag_de_la_release()
print(f"[post] la release du jour : {TAG} — le convertisseur y est tiré.")
urllib.request.urlretrieve(
    f"https://raw.githubusercontent.com/ggml-org/llama.cpp/{TAG}/convert_lora_to_gguf.py",
    CONVERTISSEUR)
rep = subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "gguf"],
                     capture_output=True, text=True)
if rep.returncode != 0:
    raise SystemExit(f"PIP-ROUGE (gguf) : {rep.stderr[-400:]}")

# 2 · la conversion (f16 — le delta, l'échelle α/r appliquée par le convertisseur)
r = subprocess.run([sys.executable, "-X", "utf8", str(CONVERTISSEUR),
                    str(DOS_ADAPTER), "--outfile", str(GGUF_ADAPTER), "--outtype", "f16"],
                   capture_output=True, text=True)
if r.returncode != 0 or not GGUF_ADAPTER.exists():
    print(r.stdout[-1500:])
    print(r.stderr[-1500:])
    raise SystemExit("ADAPTER-CONVERSION-ROUGE : le convertisseur a refusé — le journal "
                     "ci-dessus TEL QUEL (l'adapter safetensors demeure, la voie de "
                     "repli — la fusion — est nommée au plan gelé § 6.4).")
print(f"[post] l'adapter GGUF né : {GGUF_ADAPTER} ({GGUF_ADAPTER.stat().st_size / 2**20:.1f} Mio, f16).")

# 3 · LA PIÈCE AU COFFRE AVANT L'ÉVAL (le plan gelé § 4 — le relais embarqué)
try:
    CLIENT_ID, CLIENT_SECRET, REFRESH_TOKEN
except NameError:
    raise SystemExit("ORDRE-ROMPU : les identifiants du relais vivent à la cellule 3.")
ID_TEST = "1KeJIMwpsT7GshcjIJy0Sf8KzsOBjimU0"    # le /test du coffre (recoupé au labo le 27/11)
TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())


def jeton_relais(essais=3):
    import urllib.parse
    data = urllib.parse.urlencode({
        "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET,
        "refresh_token": REFRESH_TOKEN, "grant_type": "refresh_token"}).encode()
    derniere = None
    for i in range(essais):
        try:
            req = urllib.request.Request("https://oauth2.googleapis.com/token", data=data)
            with urllib.request.urlopen(req, timeout=30) as rr:
                return json.loads(rr.read().decode("utf-8"))["access_token"]
        except Exception as e:
            derniere = e
            time.sleep(10)
    raise SystemExit(f"JETON-ROUGE : le relais n'a pas donné son jeton "
                     f"({derniere.__class__.__name__}).")


def televerse_multipart(tok, nom, parent, octets_):
    import urllib.parse
    frontieres = "tr314d3k1t"
    meta = {"name": nom, "parents": [parent]}
    corps = ("--" + frontieres + "\r\n"
             "Content-Type: application/json; charset=UTF-8\r\n\r\n"
             + json.dumps(meta) + "\r\n"
             + "--" + frontieres + "\r\n"
             "Content-Type: application/octet-stream\r\n\r\n").encode("utf-8")
    corps += octets_ + ("\r\n--" + frontieres + "--\r\n").encode("utf-8")
    req = urllib.request.Request(
        "https://www.googleapis.com/upload/drive/v3/files?uploadType=multipart",
        data=corps, method="POST",
        headers={"Authorization": f"Bearer {tok}",
                 "Content-Type": f"multipart/related; boundary={frontieres}"})
    with urllib.request.urlopen(req, timeout=900) as rr:
        return json.loads(rr.read().decode("utf-8"))["id"]


tok = jeton_relais()
for piece in (GGUF_ADAPTER, REGISTRE):
    nom = f"{piece.stem}-{TS}{piece.suffix}"
    idp = televerse_multipart(tok, nom, ID_TEST, piece.read_bytes())
    print(f"[post] LA PIÈCE AU COFFRE (avant l'éval) : {nom} — id {idp}")
print("[post] L'ŒUVRE EST AU COFFRE — l'évaluation peut courir (la pièce est sauvée).")


## 7 · LE SERVING DU MODÈLE AFFINÉ — le GGUF Q6_K byte-byte de la 105 + LE DELTA `--lora` (la seule différence du bord) + LA PORTE DE PAROLE

In [ ]:
# CELLULE 7 — LE SERVING DU MODÈLE AFFINÉ (v1061 — la cellule du v1052 : les
# DEUX correctifs du jour, le terrain.9 bins_cuda + le terrain.10 bivalent
# — le v1051 — LE TERRAIN.9 DU
# 28/11 : l'emballage des releases llama.cpp a séparé binaires CUDA et cudart,
# l'asset « cudart- » ne porte PLUS les binaires — le choix pointe « -bin-
# ubuntu-cuda- » + le cudart en accompagnement, la source demeure le repli) :
# llama.cpp au runtime + le GGUF
# du bord + LA PORTE DE PAROLE (v105 — l'Entrée 451 (c) : après /health, UNE
# requête d'essai enrobée ; BORD-MUET le refus nommé — en secondes, pas en
# 27 minutes).
# byte-byte la 105 + LE DELTA --lora (l'adapter né à la cellule 5). La commande
# -m <gguf> --port 8080 -c 8192 [-ngl 99]) reconduite au poste Colab. LE BORD
# EST DÉCIDÉ PAR LE POSTE (v104 — le kit 0 clic, la demande du maître :
# « pour environnement cpu et environnement gpu ») : GPU → le GGUF Q6_K
# (6,25 Go) au cudart prébâti, -ngl 99 ; CPU → le GGUF Q4_K_M (4,68 Go — le
# plus petit du kit, le seul conseillé au bord CPU dès le verdict du v103),
# le binaire ubuntu-x64 NUE, AUCUN -ngl. LE TERRAIN.5 DU 27/11 NOMMÉ :
# « couldn't bind … 8080 » = LE PORT OCCUPÉ, PAS le binaire — le triage
# juste reconduit (le prébâti réessayé au port libre, la source NE compile
# PAS pour un port pris ; le repli qui PARLE, ligne à ligne).
REPO_GGUF = "bartowski/Qwen2.5-Coder-7B-Instruct-GGUF"
PORTS = [8080, 8081, 8090, 18080, 18081]   # les candidats : 8080 peut être occupé au poste (le terrain.5)

import json
import os
import pathlib
import socket
import subprocess
import tarfile
import time
import urllib.request
import zipfile


def gpu_presente():
    try:
        import torch
        return torch.cuda.is_available()
    except Exception:
        return False


GPU = gpu_presente()
DOS_LLAMA = pathlib.Path("/content/llama")
CHEMIN_ENDPOINT = pathlib.Path("/content/endpoint_serving.txt")


def port_libre(port):
    """La sonde au socket même — SO_REUSEADDR porté : la même tolérance que
    llama-server au bind (la sonde n'est jamais plus sévère que le serveur)."""
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        s.bind(("127.0.0.1", port))
        return True
    except OSError:
        return False
    finally:
        s.close()


def choisi_le_port(apres=None):
    """Le premier port LIBRE des candidats — après celui qui vient d'échouer
    le cas échéant (le réessai ne reprend JAMAIS le port qui a refusé)."""
    for p in PORTS:
        if apres is not None and p <= apres:
            continue
        if port_libre(p):
            return p
    return None


def menage_des_orphelins():
    """L'hygiène du terrain.5 : un llama-server orphelin d'un run précédent du
    MÊME runtime squatte le port — LUI SEUL est tué (jamais un tiers du poste)."""
    r = subprocess.run(["pkill", "-9", "-f", "llama-server"], capture_output=True)
    tue = r.returncode == 0
    if tue:
        time.sleep(2)      # le relâchement du port par le noyau
    return tue


def asset_de_la_release():
    """La release PRÉBATTIE du jour : les tags b portent les binaires (les tags v
    n'en portent pas — vérifié au bac : 38 assets au b12810). Le choix :
    GPU → le cudart (le runtime CUDA embarqué, tolérant au driver du poste) ;
    CPU → le ubuntu-x64 nu. Retour None si l'API ou le choix manque — la source
    prend le relais (le repli NOMMÉ)."""
    try:
        with urllib.request.urlopen(
                "https://api.github.com/repos/ggml-org/llama.cpp/releases?per_page=12",
                timeout=30) as r:
            releases = json.loads(r.read().decode("utf-8"))
    except Exception as e:
        print(f"[serving] l'API des releases a manqué ({e.__class__.__name__}) — la SOURCE prend le relais.")
        return None
    for rel in releases:
        nu, bins_cuda, cudarts = None, [], []
        for a in rel["assets"]:
            n = a["name"].lower()
            if "ubuntu" not in n or "x64" not in n or not n.endswith((".zip", ".tar.gz")):
                continue
            exclus = ("arm64", "s390x", "vulkan", "rocm", "sycl", "openvino", "opencl")
            if any(x in n for x in exclus):
                continue
            # LE TERRAIN.9 (28/11) : l'emballage a SÉPARÉ binaires et cudart —
            # l'asset « cudart-… » ne porte PLUS les binaires (3 libs seules,
            # mesuré au bac) ; les binaires CUDA vivent à « …-bin-ubuntu-cuda-… ».
            if n.startswith("cudart-") and "cuda" in n:
                cudarts.append(a)
            elif "cuda" in n and n.startswith("llama-") and "-bin-ubuntu-cuda-" in n:
                bins_cuda.append(a)
            elif "cuda" not in n and not n.startswith("cudart-"):
                nu = a
        if GPU and bins_cuda:
            # les BINAIRES CUDA-12 d'abord (le cudart CUDA-12 en accompagnement,
            # plus tolérant au driver du poste que le 13)
            bin_ = next((a for a in bins_cuda if "cuda-12" in a["name"].lower()), bins_cuda[0])
            lib = next((a for a in cudarts if "cuda-12" in a["name"].lower()),
                       cudarts[0] if cudarts else None)
            return rel["tag_name"], bin_, lib
        if not GPU and nu is not None:
            return rel["tag_name"], nu, None
    return None


def extrais():
    """L'extraction du prébâti : les tags b sortent le dossier <llama-bXXXXX> à
    plats (binaires + libs .so frères — vérifié au bac)."""
    tag_asset = asset_de_la_release()
    if tag_asset is None:
        return None
    tag, asset, asset_lib = tag_asset
    print(f"[serving] la release prébattue {tag} : {asset['name']} "
          f"({asset['size'] / 2**20:.0f} Mio)"
          + (f" + le cudart {asset_lib['name']} ({asset_lib['size'] / 2**20:.0f} Mio)"
             if asset_lib else ""))
    for ass in (asset, asset_lib):
        if ass is None:
            continue
        archive = pathlib.Path("/content") / ass["name"]
        if archive.exists():
            archive.unlink()
        urllib.request.urlretrieve(ass["browser_download_url"], archive)
        if ass["name"].endswith(".zip"):
            with zipfile.ZipFile(archive) as z:
                z.extractall(DOS_LLAMA if ass is asset else DOS_LLAMA / "cudart")
        else:
            with tarfile.open(archive) as t:
                try:
                    t.extractall(DOS_LLAMA if ass is asset else DOS_LLAMA / "cudart",
                                 filter="data")
                except TypeError:
                    t.extractall(DOS_LLAMA if ass is asset else DOS_LLAMA / "cudart")
    trouves = list(DOS_LLAMA.rglob("llama-server"))
    return trouves[0] if trouves else None


def compile_la_source():
    """LE REPLI QUI PARLE : le terrain.5 nomma le v100 muet (la compilation
    capturée, les minutes sans journal) — chaque ligne coule au poste à
    présent, la cible llama-server seule bâtie (le chantier raccourci)."""
    print("[serving] la compilation depuis la source — LE CHANTIER QUI PARLE "
          "(5-15 min aux vCPU du poste, le café est nommé ; chaque ligne ci-dessous).")
    commande = ("git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp && "
                "cmake -S /content/llama.cpp -B /content/llama.cpp/build"
                + (" -DGGML_CUDA=ON" if GPU else "") + " && "
                "cmake --build /content/llama.cpp/build --config Release "
                "--target llama-server -j$(nproc)")
    debut = time.time()
    proc = subprocess.Popen(["bash", "-c", commande], stdout=subprocess.PIPE,
                            stderr=subprocess.STDOUT, text=True, bufsize=1,
                            encoding="utf-8", errors="replace")
    n_lignes = 0
    for ligne in proc.stdout:
        n_lignes += 1
        print(ligne, end="")
    code = proc.wait()
    print(f"[serving] la compilation close : code {code}, {n_lignes} lignes, "
          f"{(time.time() - debut) / 60:.1f} min.")
    if code != 0:
        raise SystemExit(f"[ROUGE] la compilation a échoué (code {code}) — le journal ci-dessus TEL QUEL.")
    return pathlib.Path("/content/llama.cpp/build/bin/llama-server")


def lance_serving(binaire, gguf, port):
    """Le lancement + la porte de santé (240 × 5 s) au PORT choisi. Les libs
    .so frères du binaire prébâti portent au LD_LIBRARY_PATH du poste."""
    env = dict(os.environ)
    dos_cudart = DOS_LLAMA / "cudart"
    chemins_lib = [str(binaire.parent)]
    for sous in dos_cudart.rglob("*"):
        if sous.is_dir() and (sous / "libcudart.so.12").exists():
            chemins_lib.append(str(sous))      # le terrain.9 : le cudart EXTRACT à part
    env["LD_LIBRARY_PATH"] = ":".join(chemins_lib) + (
        ":" + env["LD_LIBRARY_PATH"] if env.get("LD_LIBRARY_PATH") else "")
    args_serving = [str(binaire), "-m", str(gguf), "--host", "127.0.0.1",
                    "--port", str(port), "-c", "8192"]
    if GPU:
        args_serving += ["-ngl", "99"]
    args_serving += ["--lora", str(pathlib.Path("/content/adapter-entrainement-v106.gguf"))]
    journal = open("/content/llama-server.log", "w", encoding="utf-8")
    proc = subprocess.Popen(args_serving, stdout=journal, stderr=subprocess.STDOUT, env=env)
    for _ in range(240):
        time.sleep(5)
        if proc.poll() is not None:
            break
        try:
            with urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=10) as r:
                if r.status == 200:
                    journal.close()
                    return proc, True
        except Exception:
            pass
    journal.close()
    return proc, False


def lit_journal():
    """Le journal du serveur, TEL QUEL (le terrain.5 fut lu par ce biais)."""
    try:
        return open("/content/llama-server.log", encoding="utf-8", errors="replace").read()
    except OSError:
        return ""


# LE BORD DÉCIDÉ PAR LE POSTE (v104) :
QUANT = "Q6_K" if GPU else "Q4_K_M"
print(f"[serving] le bord du run : {'GPU' if GPU else 'CPU'} — le GGUF {QUANT} "
      f"({'6,25 Go' if GPU else '4,68 Go — le plus petit du kit, le seul conseillé au bord CPU'}).")

# 1 · llama-server : la release prébattue d'abord, la source compilée en repli nommé
binaire = None
if DOS_LLAMA.exists():
    trouves = list(DOS_LLAMA.rglob("llama-server"))
    if trouves:
        binaire = trouves[0]
if binaire is None:
    try:
        binaire = extrais()
    except Exception as e:
        print(f"[serving] le prébâti a manqué ({e.__class__.__name__}) — la SOURCE prend le relais.")
libelle = str(binaire) if binaire is not None else "la SOURCE d'abord"
print(f"[serving] llama-server : {libelle}")

# 2 · le GGUF du modèle choisi (le vocabulaire vit au harnais — recoupé au chargement)
from huggingface_hub import hf_hub_download

gguf = hf_hub_download(repo_id=REPO_GGUF, filename=f"Qwen2.5-Coder-7B-Instruct-{QUANT}.gguf")
print(f"[serving] le GGUF {QUANT} : {gguf} "
      f"({pathlib.Path(gguf).stat().st_size / 2**30:.2f} Gio)")

# 3 · L'HYGIÈNE DU PORT (le terrain.5) : les orphelins d'un run précédent tués,
#     le port LIBRE choisi — AUCUN port n'est imposé au poste
if menage_des_orphelins():
    print("[serving] un llama-server orphelin d'un run précédent a été tué (l'hygiène du terrain.5).")
PORT = choisi_le_port()
if PORT is None:
    raise SystemExit(f"PORT-OCCUPÉ : aucun candidat n'est libre ({PORTS}) — "
                     "le journal du poste le dira ; AUCUN serving ne court (fail-closed).")
if PORT != 8080:
    print(f"[serving] le 8080 est occupé au poste — le port libre {PORT} pris (le terrain.5 nommé).")

# 4 · le serving + la porte de santé ; LE TRIAGE DU TERRAIN.5 : « couldn't bind »
#     = LE PORT, PAS LE BINAIRE — le prébâti réessayé au port suivant, la source
#     NE COMPILE PAS pour un port pris ; le binaire mort d'une faute qui est
#     sienne (glibc, driver) → la source prend le relais (le repli NOMMÉ)
proc_serving, sain = lance_serving(binaire, gguf, PORT) if binaire else (None, False)
if not sain and binaire is not None and "couldn't bind" in lit_journal():
    print("[serving] le port a refusé (couldn't bind) — le PRÉBÂTI EST SAIN, c'est le port : "
          "le réessai au port suivant, la source NE compile PAS (le triage du terrain.5).")
    if menage_des_orphelins():
        print("[serving] un second orphelin tué au réessai.")
    PORT = choisi_le_port(apres=PORT) or choisi_le_port()
    if PORT is None:
        raise SystemExit(f"PORT-OCCUPÉ : plus aucun candidat libre ({PORTS}) — "
                         "le journal du poste le dira.")
    print(f"[serving] le réessai au port {PORT}.")
    proc_serving, sain = lance_serving(binaire, gguf, PORT)
if not sain:
    if binaire is not None:
        print("[serving] le prébâti n'est pas sain au poste — le journal TEL QUEL :")
        print(lit_journal()[-1500:], end="")
        print("[serving] la SOURCE prend le relais (le repli NOMMÉ).")
    binaire = compile_la_source()
    if menage_des_orphelins():
        print("[serving] un orphelin tué avant le serving de la source.")
    PORT = choisi_le_port(apres=PORT) or choisi_le_port()
    if PORT is None:
        raise SystemExit(f"PORT-OCCUPÉ : plus aucun candidat libre ({PORTS}) après la compilation.")
    proc_serving, sain = lance_serving(binaire, gguf, PORT)
if not sain:
    print(lit_journal()[-2500:], end="")
    raise SystemExit("[ROUGE] le serving n'est pas sain — le journal ci-dessus TEL QUEL "
                     "(un échec nommé vaut plus qu'un vert muet).")
# 5 bis · LA PORTE DE PAROLE (v105 — la garde nouvelle, l'Entrée 451 (c)) :
#     après /health, UNE requête d'essai ENROBÉE — le modèle doit DIRE un
#     mot avant que la batterie ne coure. BORD-MUET le refus nommé : le
#     poste saurait en ~30 secondes, pas en 27 minutes (la leçon du v104 au
#     bord CPU : les 41 requêtes muettes y coururent 27,1 minutes). Le
#     payload d'essai est celui de LA SONDE DU KIT même (moteur_campagne.py,
#     la pièce vivante — jamais une copie) : la porte et l'étape 3 de la
#     batterie disent le même langage.
import sys
sys.path.insert(0, "/content/kit")
from moteur_campagne import SONDE_PAYLOAD, enveloppe

essai_corps = json.dumps({"prompt": enveloppe(SONDE_PAYLOAD), "n_predict": 1,
                          "n_probs": 8, "temperature": 0.0,
                          "cache_prompt": True}).encode("utf-8")
req_essai = urllib.request.Request(f"http://127.0.0.1:{PORT}/completion",
                                   data=essai_corps,
                                   headers={"Content-Type": "application/json"})
probs_essai, rep_essai = [], {}
try:
    with urllib.request.urlopen(req_essai, timeout=180) as r:
        rep_essai = json.loads(r.read().decode("utf-8"))
    # LE TERRAIN.10 (28/11) : la clé « probs » est devenue « top_logprobs » à
    # b12823 (mesuré au bord au diagnostic du 28/11) — les DEUX formats admis.
    entree_essai = rep_essai["completion_probabilities"][0]
    probs_essai = (entree_essai.get("probs") or entree_essai.get("top_logprobs") or [])
except Exception as e:
    print(lit_journal()[-1500:], end="")
    raise SystemExit(f"BORD-MUET : la requête d'essai enrobée n'a pas rendu sa parole "
                     f"({e.__class__.__name__}) — la batterie NE COURT PAS (le refus "
                     f"vient à la porte, en secondes — pas en 27 minutes). Le journal "
                     f"du serving ci-dessus TEL QUEL.")
if not probs_essai:
    print(lit_journal()[-1500:], end="")
    raise SystemExit("BORD-MUET : la requête d'essai enrobée n'a rendu AUCUNE parole "
                     "(aucune probabilité rendue — le modèle finit son tour avant "
                     "d'écrire UN jeton, la cause nommée à l'Entrée 451) : la batterie "
                     "NE COURT PAS (le refus vient à la porte, en secondes — pas en "
                     "27 minutes). Le journal du serving ci-dessus TEL QUEL.")
mot_essai = rep_essai["completion_probabilities"][0].get("content", "")
print(f"[serving] LA PORTE DE PAROLE : VERT — le modèle dit son premier jeton "
      f"« {mot_essai!r} » (la parole est ouverte, la batterie peut courir).")

CHEMIN_ENDPOINT.write_text(f"http://127.0.0.1:{PORT}\n", encoding="utf-8")
print(f"[serving] http://127.0.0.1:{PORT}/health : VERT — le serving du modèle choisi est vivant.")
print(f"[serving] l'endpoint PORTÉ au fichier {CHEMIN_ENDPOINT} — la cellule 6 (LA BATTERIE) "
      "le lit (top-n 64).")


## 8 · LA BATTERIE 0 CLIC DU MODÈLE AFFINÉ — les SIX étapes du plan gelé, l'instrument de la 105 TEL QUEL

In [ ]:
# CELLULE 8 — LA BATTERIE 0 CLIC DU MODÈLE AFFINÉ : les SIX étapes du plan gelé
# (v105 : LA SONDE DES DEUX ENVELOPPES ouvre la marche — l'Entrée 451 au
# poste même ; BORD-MUET et PAROLE-REFUSÉE y refusent fail-closed).
# d'un trait (le patron de la cellule de secours 5, l'Entrée 448, porté à
# cellule du kit — la conséquence du terrain.8 : la GUI est retirée, ce que
# le runtime ne peut pas attendre, le kit ne le lui demande plus). Le
# journal vit EN DIRECT, l'horloge tient chaque étape (le rapport du poste
# y vivra), un ROUGE ARRÊTE la marche (fail-closed) — « Tout exécuter »
# s'arrête à la cellule fautive, les suivantes ne courent pas (la loi du
# runtime, l'Entrée 449). LE MOTEUR PORTE TOUT le poids du geste : la
# batterie ne fait QUE le lancer et tenir l'horloge.
import pathlib
import subprocess
import sys
import time

MOTEUR = "/content/kit/moteur_campagne.py"
DOS_MOTEUR = "/content/kit"
CHEMIN_ENDPOINT = pathlib.Path("/content/endpoint_serving.txt")
HORLOGE = pathlib.Path("/content/horloge-batterie.txt")

if not pathlib.Path(MOTEUR).exists():
    raise SystemExit("CLI-ABSENT : aucun /content/kit/moteur_campagne.py — "
                     "la cellule 3 (les kits au coffre) d'abord.")
if not CHEMIN_ENDPOINT.exists():
    raise SystemExit("CLI-ABSENT : aucun /content/endpoint_serving.txt — "
                     "la cellule 7 (le serving affiné) d'abord.")
endpoint = CHEMIN_ENDPOINT.read_text(encoding="utf-8").strip()
print(f"[batterie] l'endpoint lu du serving : {endpoint}")
GPU_BORD = False
try:
    import torch
    GPU_BORD = torch.cuda.is_available()
except Exception:
    GPU_BORD = False
print(f"[batterie] le bord : {'GPU' if GPU_BORD else 'CPU'} — la campagne y court TELLE QUELLE "
      "(le registre porte le mode B, la mesure, jamais maquillée).")
if GPU_BORD:
    print("[batterie] l'estimation du kit : de quelques dizaines de minutes à ~2 heures au MODE-B "
          "sur T4 selon la charge — le journal vit en direct ci-dessous, NE TOUCHE À RIEN.")
else:
    print("[batterie] L'ESTIMATION HONNÊTE du bord CPU : PLUSIEURS HEURES (la mesure seule dira) —")
    print("[batterie] le journal vit en direct ci-dessous, NE TOUCHE À RIEN, la session demeure ouverte.")
HORLOGE.write_text("", encoding="utf-8")

ETAPES = [
    ("1 · Vérifier le kit (le manifeste)", ["--verifier-kit"]),
    ("2 · Le gate (la sonde du décodeur client)", ["--gate"]),
    ("3 · LA SONDE DES DEUX ENVELOPPES (l'Entrée 451 — BORD-MUET le refus nommé)",
     ["--sonde-bords", "--mode", "B", "--url", endpoint, "--nprobs", "64"]),
    ("4 · Le canari (la violation ordonnée refusée)",
     ["--canari", "--mode", "B", "--url", endpoint, "--nprobs", "64"]),
    ("5 · LA CAMPAGNE (éditer 18 + ajouter 18 + la seconde famille)",
     ["--campagne", "--mode", "B", "--url", endpoint, "--nprobs", "64"]),
    ("6 · Le zip de preuves", ["--zip"]),
]

for titre, arguments in ETAPES:
    print(f"\n[batterie] === L'ÉTAPE : {titre} ===", flush=True)
    debut = time.time()
    code = subprocess.call([sys.executable, "-X", "utf8", MOTEUR] + arguments,
                           cwd=DOS_MOTEUR)
    duree = time.time() - debut
    with open(HORLOGE, "a", encoding="utf-8") as f:
        f.write(f"{titre}\t{duree:.0f}\n")
    if code != 0:
        raise SystemExit(f"[batterie] ROUGE (code {code}) — {titre}. La marche s'arrête "
                         "(fail-closed) ; le journal TEL QUEL ci-dessus nomme l'écart.")
    print(f"[batterie] VERT — {titre} ({duree / 60:.1f} min)", flush=True)

print("\n[batterie] LES SIX ÉTAPES VERTES — la cellule 9 porte le zip au coffre (0 clic).")


## 9 · LE RETOUR AU COFFRE — **0 CLIC** : le zip d'entraînement (l'adapter + le registre + la mesure du moteur) téléversé au `/test`, recopé byte-byte

In [ ]:
# CELLULE 9 — LE RETOUR AU COFFRE, 0 CLIC : LE ZIP D'ENTRAÎNEMENT (l'adapter
# safetensors + le registre + la courbe + LE ZIP DU MOTEUR + le rapport du
# poste) téléversé au /test du coffre par LE RELAIS EMBARQUÉ, RECOPÉ
# byte-byte PAR RELECTURE. Le pilote du labo rapatrie le preuves-campagne-*
# le plus récent du kit — le présent porte TOUT (l'œuvre ET la mesure).
import hashlib
import json
import pathlib
import time
import urllib.parse
import urllib.request
import zipfile

try:
    CLIENT_ID, CLIENT_SECRET, REFRESH_TOKEN
except NameError:
    raise SystemExit("ORDRE-ROMPU : les identifiants du relais vivent à la cellule 3.")
ID_TEST = "1KeJIMwpsT7GshcjIJy0Sf8KzsOBjimU0"
DOS_KIT = pathlib.Path("/content/kit")
DOS_ADAPTER = pathlib.Path("/content/adapter-entrainement-v106")
GGUF_ADAPTER = pathlib.Path("/content/adapter-entrainement-v106.gguf")
REGISTRE = pathlib.Path("/content/registre-entrainement-v106.json")
RAPPORT = pathlib.Path("/content/rapport-poste.txt")
HORLOGE = pathlib.Path("/content/horloge-batterie.txt")


def sha256f(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for morceau in iter(lambda: f.read(1 << 20), b""):
            h.update(morceau)
    return h.hexdigest()


def jeton_relais(essais=3):
    data = urllib.parse.urlencode({
        "client_id": CLIENT_ID, "client_secret": CLIENT_SECRET,
        "refresh_token": REFRESH_TOKEN, "grant_type": "refresh_token"}).encode()
    derniere = None
    for i in range(essais):
        try:
            req = urllib.request.Request("https://oauth2.googleapis.com/token", data=data)
            with urllib.request.urlopen(req, timeout=30) as r:
                return json.loads(r.read().decode("utf-8"))["access_token"]
        except Exception as e:
            derniere = e
            time.sleep(10)
    raise SystemExit(f"JETON-ROUGE ({derniere.__class__.__name__}).")


def televerse_multipart(tok, nom, parent, octets_):
    frontieres = "tr314d3k1t"
    meta = {"name": nom, "parents": [parent]}
    corps = ("--" + frontieres + "\r\n"
             "Content-Type: application/json; charset=UTF-8\r\n\r\n"
             + json.dumps(meta) + "\r\n"
             + "--" + frontieres + "\r\n"
             "Content-Type: application/octet-stream\r\n\r\n").encode("utf-8")
    corps += octets_ + ("\r\n--" + frontieres + "--\r\n").encode("utf-8")
    req = urllib.request.Request(
        "https://www.googleapis.com/upload/drive/v3/files?uploadType=multipart",
        data=corps, method="POST",
        headers={"Authorization": f"Bearer {tok}",
                 "Content-Type": f"multipart/related; boundary={frontieres}"})
    with urllib.request.urlopen(req, timeout=1200) as r:
        return json.loads(r.read().decode("utf-8"))["id"]


def relit_le_coffre(tok, file_id):
    req = urllib.request.Request(
        f"https://www.googleapis.com/drive/v3/files/{file_id}?alt=media",
        headers={"Authorization": f"Bearer {tok}"})
    with urllib.request.urlopen(req, timeout=1200) as r:
        return r.read()


# 1 · le zip du moteur (la batterie l'a rendu à la cellule 8)
zips_moteur = sorted(DOS_KIT.glob("preuves-campagne-2*.zip"), key=lambda p: p.stat().st_mtime)
if not zips_moteur:
    raise SystemExit("ZIP-ABSENT : aucun preuves-campagne du moteur au kit — "
                     "la cellule 8 (LA BATTERIE) d'abord.")
zip_moteur = zips_moteur[-1]

# 2 · le rapport du poste clos (l'horloge de la batterie y vit)
if not RAPPORT.exists():
    raise SystemExit("ORDRE-ROMPU : aucun rapport-poste.txt — la cellule 1 d'abord.")
if HORLOGE.exists():
    with open(RAPPORT, "a", encoding="utf-8") as f:
        f.write("\nL'HORLOGE DE LA BATTERIE (les six étapes du plan gelé) :\n")
        for l in HORLOGE.read_text(encoding="utf-8").strip().splitlines():
            titre_etape, secondes = l.split("\t")
            f.write(f"  {titre_etape} — {float(secondes) / 60:.1f} min\n")

# 3 · LE ZIP D'ENTRAÎNEMENT (l'œuvre ET la mesure, une seule pièce)
TS = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
NOM_ZIP = f"preuves-campagne-entrainement-v106-{TS}.zip"
CHEMIN_ZIP = DOS_KIT / NOM_ZIP
with zipfile.ZipFile(CHEMIN_ZIP, "zip", compresslevel=6) as z:
    z.write(zip_moteur, "moteur/" + zip_moteur.name)
    z.write(RAPPORT, "rapport-poste.txt")
    z.write(REGISTRE, "registre-entrainement-v106.json")
    for p in sorted(DOS_ADAPTER.rglob("*")):
        if p.is_file():
            z.write(p, "adapter-safetensors/" + str(p.relative_to(DOS_ADAPTER)))
    if GGUF_ADAPTER.exists():
        z.write(GGUF_ADAPTER, GGUF_ADAPTER.name)
empreinte = sha256f(CHEMIN_ZIP)
print(f"[retour] la pièce : {NOM_ZIP} ({CHEMIN_ZIP.stat().st_size / 2**20:.1f} Mio)")
print(f"[retour] sha256 : {empreinte}")
with open(RAPPORT, "a", encoding="utf-8") as f:
    f.write(f"\nLE ZIP D'ENTRAÎNEMENT : {NOM_ZIP}\nsha256 : {empreinte}\n")

# 4 · LE TÉLÉVERSEMENT 0 CLIC + LE RECOP byte-byte
print("[retour] LE JETON RELAIS se frappe — le téléversement court (aucune fenêtre).")
tok = jeton_relais()
id_zip = televerse_multipart(tok, NOM_ZIP, ID_TEST, CHEMIN_ZIP.read_bytes())
print(f"[retour] le zip téléversé au /test du coffre : id {id_zip}")
time.sleep(3)
relu = relit_le_coffre(tok, id_zip)
if hashlib.sha256(relu).hexdigest() != empreinte:
    raise SystemExit("RECOP-ROUGE : la pièce relue du coffre diverge — le journal TEL QUEL.")
print(f"[retour] RECOP byte-byte VERT — {NOM_ZIP} vit au /test du coffre.")
id_rapport = televerse_multipart(tok, f"rapport-poste-entrainement-v106-{TS}.txt",
                                 ID_TEST, RAPPORT.read_bytes())
print(f"[retour] le rapport du poste téléversé : id {id_rapport}")
print(f"[retour] LE RUN D'ENTRAÎNEMENT EST CLOSE — les liens vivent ci-dessus, "
      f"les verdicts TELS QUELS au /test (le labo les lira).")


## 10 · LE BILAN DU RUN — le tableau de l'horloge, les pièces rendues, le run clos

In [ ]:
# CELLULE 10 — LE BILAN DU RUN 0 CLIC : le tableau de l'horloge, les pièces
# rendues, le serving tué (le port rendu au poste), le run clos.
import pathlib
import subprocess

HORLOGE = pathlib.Path("/content/horloge-batterie.txt")

print("LE BILAN DU RUN 0 CLIC (kit Colab v106 ENTRAÎNEMENT) — l'horloge de la batterie :")
print()
print("  l'étape                                                        la durée")
print("  " + "-" * 76)
if HORLOGE.exists():
    for l in HORLOGE.read_text(encoding="utf-8").strip().splitlines():
        titre, secondes = l.split("\t")
        print(f"  {titre:<62} {float(secondes) / 60:>8.1f} min")
print("  " + "-" * 76)

for z in sorted(pathlib.Path("/content/kit").glob("preuves-campagne-*.zip"),
                key=lambda p: p.stat().st_mtime):
    print(f"[bilan] la pièce du kit : {z.name} ({z.stat().st_size} o) — "
          "déposée au /test du coffre (la cellule 7, les liens y sont imprimés)")
rapport = pathlib.Path("/content/rapport-poste.txt")
if rapport.exists():
    print(f"[bilan] le rapport du poste : {rapport} (déposé avec le zip au /test du coffre)")

r = subprocess.run(["pkill", "-9", "-f", "llama-server"], capture_output=True)
if r.returncode == 0:
    print("[bilan] le serving llama-server tué — le port rendu au poste.")
else:
    print("[bilan] aucun llama-server à tuer — le poste est déjà propre.")
print()
print("[bilan] LE RUN D'ENTRAÎNEMENT EST CLOSE : le zip (l'adapter + la mesure) + le rapport")
print("[bilan] du poste vivent au /test du coffre. Les verdicts (tenue ou manquée, jamais")
print("[bilan] habillée) y sont TELS QUELS — le labo les lira ; la comparaison pré/post")
print("[bilan] s'y conduira (la campagne 105 est la ligne de base).")
